In [ ]:
import os
from neo4j import GraphDatabase

NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("Connected to Neo4j")

In [ ]:
CYPHER_HUBS = """
MATCH (n)-[r]-()
WITH n, count(r) AS degree
WHERE degree > 3
RETURN elementId(n) AS node_id,
       n.name       AS name,
       n.type       AS type,
       degree
ORDER BY degree DESC
LIMIT 100
"""

with driver.session() as session:
    hubs = session.run(CYPHER_HUBS).data()

print(f"Found {len(hubs)} hubs (>=5 connections)\n")
print(f"{'Rank':<5} {'Degree':<8} {'Type':<20} {'Name'}")
print("-" * 70)
for i, h in enumerate(hubs, 1):
    print(f"{i:<5} {h['degree']:<8} {h['type']:<20} {h['name']}")

In [ ]:
CYPHER_CROSS_DOC = """
MATCH (n)
WHERE elementId(n) = $node_id

MATCH (n)-[r]-(neighbor)

WITH n, r, neighbor,
     [cid IN n.chunk_ids        | substring(cid, 0, size(cid)-5)] AS center_docs,
     [cid IN coalesce(neighbor.chunk_ids, []) | substring(cid, 0, size(cid)-5)] AS neighbor_docs

WITH n, r, neighbor, center_docs, neighbor_docs,
     [doc IN neighbor_docs WHERE NOT doc IN center_docs] AS new_docs

WHERE size(new_docs) > 0

RETURN
    n.name          AS center_name,
    n.type          AS center_type,
    neighbor.name   AS neighbor_name,
    neighbor.type   AS neighbor_type,
    new_docs        AS new_docs,
    size(new_docs)  AS new_docs_count,
    r.type          AS relation,
    CASE WHEN startNode(r) = n THEN 'OUT' ELSE 'IN' END AS direction
ORDER BY new_docs_count DESC
"""

raw_results = {}  # {node_id: [rows]}

with driver.session() as session:
    for hub in hubs:
        nid   = hub["node_id"]
        rows  = session.run(CYPHER_CROSS_DOC, node_id=nid).data()
        if rows:
            raw_results[nid] = rows

print(f"Results for {len(raw_results)} / {len(hubs)} hubs contain cross-doc neighbors")

In [ ]:
from collections import defaultdict

# triple: (center_name, center_type, relation, neighbor_name, neighbor_type)
# + how many new documents it brings

triples = defaultdict(lambda: {"count": 0, "new_docs_total": 0, "new_docs_examples": set()})

for nid, rows in raw_results.items():
    for row in rows:
        cn   = row["center_name"]
        ct   = row["center_type"]   or "?"
        nn   = row["neighbor_name"]
        nt   = row["neighbor_type"] or "?"
        rel  = row["relation"]
        nd   = row["new_docs_count"]
        dirn = row["direction"]

        if dirn == "OUT":
            key = (cn, ct, rel, nn, nt)
        else:
            key = (nn, nt, rel, cn, ct)

        triples[key]["count"]          += 1
        triples[key]["new_docs_total"] += nd
        for doc in row["new_docs"]:
            triples[key]["new_docs_examples"].add(doc)

print(f"Unique cross-document triples: {len(triples)}")

In [ ]:
import pandas as pd

rows_out = []
for (cn, ct, rel, nn, nt), meta in triples.items():
    rows_out.append({
        "head":           f"{ct}: {cn}",
        "relation":       rel,
        "tail":           f"{nt}: {nn}",
        "occurrences":    meta["count"],
        "new_docs_total": meta["new_docs_total"],
        "new_docs_count": len(meta["new_docs_examples"]),
    })

df = pd.DataFrame(rows_out).sort_values(
    ["new_docs_count", "new_docs_total"], ascending=False
).reset_index(drop=True)

print(f"Top 30 cross-document triples (most new documents):\n")
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", 30)
display(df.head(30))

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-5.1"

client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client ready")

In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT = """You are building an evaluation dataset for Graph RAG systems.
Your task: given knowledge graph triples, generate ONE question-answer pair that is IMPOSSIBLE to answer without graph traversal.

═══════════════════════════════════════════
CORE RULE: THE ANSWER MUST CONTAIN ≥2 DISTINCT FACTS FROM SEPARATE DOCUMENTS
═══════════════════════════════════════════
The answer must follow this exact structure:
  FACT_1 (from documents about [topic A]): <one specific, verifiable sentence>
  FACT_2 (from documents about [topic B]): <one specific, verifiable sentence>
  SYNTHESIS: <one sentence connecting the two facts>

If you cannot fill FACT_1 and FACT_2 with SPECIFIC facts from SEPARATE document groups → return {"skip": true}.
A "specific fact" means: a concrete action, decision, number, or named outcome — NOT a description of a relationship.

═══════════════════════════════════════════
CHOOSE ONE ARCHETYPE:
═══════════════════════════════════════════

A) ENUMERATION
   Question form: "Wobec ilu / które [X] [podmiot opisany pośrednio] [działanie]?"
   Answer: a list of ≥3 items, each from a different document group.
   Vanilla fails because: no single document lists all items together.

B) ATTRIBUTION CHAIN
   Question form: "Który [X] był jednocześnie: (a) [fakt z doc_group_1] oraz (b) [fakt z doc_group_2]?"
   Answer: FACT_1 from one document group + FACT_2 from another, both pointing to the same entity.
   Vanilla fails because: no document states both facts together.

C) CROSS-DOMAIN CONTRAST
   Question form: "Czym różni się [działanie podmiotu A] od [działania podmiotu B] w kontekście [wspólny mianownik]?"
   Answer: FACT_1 about subject A from doc_group_X + FACT_2 about subject B from doc_group_Y.
   Vanilla fails because: subjects A and B appear in semantically distant documents.

═══════════════════════════════════════════
QUESTION RULES:
═══════════════════════════════════════════
1. NEVER use real names (no "Putin", "Macron", "Białoruś", "Rosja", "NATO") — use role descriptions only
2. The question topic must be SEMANTICALLY DISTANT from the entity domains in the triples
   Example: if triples are about geopolitics → question framed around "decision-making patterns" not "war"
3. Embedding the question and searching semantically must NOT retrieve the answer documents
4. Both question and answer MUST be in POLISH

═══════════════════════════════════════════
VANILLA FAILURE TEST (apply before returning):
═══════════════════════════════════════════
Extract 3 keywords from your question.
Ask: "Would a dense vector search on these 3 keywords return documents containing BOTH facts?"
If YES → redesign the question.
The only valid path to the answer: graph.traverse(entity) → related_entities → separate_documents → combine facts.

═══════════════════════════════════════════
FORBIDDEN patterns (vanilla RAG wins these):
═══════════════════════════════════════════
✗ Narrative chains: "Jak X wpłynął na Y, co doprowadziło do Z?" — one analysis article covers this
✗ Answers that are paragraphs of political/economic analysis with no concrete facts
✗ Answers describing the reasoning process ("łącząc oba wątki...", "po połączeniu informacji...")
✗ Any question where the answer could be written without knowing specific facts from specific documents

═══════════════════════════════════════════
RETURN ONLY valid JSON:
═══════════════════════════════════════════
{
  "archetype": "ENUMERATION" | "ATTRIBUTION" | "CONTRAST",
  "question": "...",
  "answer": {
    "fact_1": {"text": "...", "doc_group": "documents about [topic]"},
    "fact_2": {"text": "...", "doc_group": "documents about [topic]"},
    "synthesis": "..."
  },
  "documents_needed": ["why doc_group_1 is needed", "why doc_group_2 is needed"],
  "vanilla_attack_vector": "3 keywords a dense search would use — explain why they fail",
  "skip": false
}"""


def build_user_prompt(triples_subset: list[dict]) -> str:
    lines = []
    doc_groups = {}
    domains = set()

    for t in triples_subset:
        line = (
            f"  ({t['head']}) --[{t['relation']}]--> ({t['tail']})"
            f"  [bridges {t['new_docs_count']} new docs]"
        )
        lines.append(line)

        for field in ("head", "tail"):
            name = t[field].split(": ", 1)[-1] if ": " in t[field] else t[field]
            if name not in doc_groups:
                doc_groups[name] = t.get("new_docs_count", 0)

        for field in ("head_type", "tail_type"):
            if t.get(field):
                domains.add(t[field])

    # Top entities ranked by cross-doc reach — these are the graph hubs
    top_entities = sorted(doc_groups.items(), key=lambda x: -x[1])[:6]
    entity_hint = "\n".join(
        f"  [{i+1}] {name} — bridges {d} document groups"
        for i, (name, d) in enumerate(top_entities)
    )

    domain_hint = ", ".join(sorted(domains)) if domains else "unknown"

    # Suggest which archetype fits based on triple structure
    n_triples = len(triples_subset)
    hub_entities = [name for name, d in top_entities if d >= 4]
    archetype_hint = ""
    if len(hub_entities) == 1:
        archetype_hint = "→ ENUMERATION fits best: one central entity connects to many document groups"
    elif len(hub_entities) == 2:
        archetype_hint = "→ CONTRAST fits best: two hub entities can be compared across their respective document groups"
    else:
        archetype_hint = "→ ATTRIBUTION fits best: find an entity that satisfies two independent facts from different triples"

    return f"""Generate a cross-document QA pair from these knowledge graph triples.

TRIPLES (each relation bridges documents not connected by keywords):
{chr(10).join(lines)}

ENTITIES BY CROSS-DOCUMENT REACH:
{entity_hint}

Entity domains present: {domain_hint}
{archetype_hint}

CRITICAL CONSTRAINTS:
- Do NOT use any of the entity names verbatim in the question
- The question's semantic topic must differ from: {domain_hint}
- FACT_1 and FACT_2 in the answer must come from DIFFERENT rows in the triples above
- If the only answer you can produce is a narrative paragraph → return {{"skip": true}}

Return only JSON."""

In [ ]:
CYPHER_DEGREES = """
MATCH (n)-[r]-()
WITH n, count(r) AS degree
WHERE degree > 5
RETURN elementId(n) AS node_id, n.name AS name, degree
"""
with driver.session() as session:
    degree_map = {
        r["name"]: r["degree"]
        for r in session.run(CYPHER_DEGREES).data()
    }

SUPER_HUB_THRESHOLD = 40

SPECIFIC_TYPES = {"Osoba", "Wydarzenie", "Koncepcja", "Zjawisko gospodarcze",
                  "Porozumienie", "Instytucja", "Organizacja"}


def is_specific(entity_str: str, type_str: str) -> bool:
    """Whether the entity is specific enough (not a super-hub)."""
    name = entity_str.split(": ", 1)[-1] if ": " in entity_str else entity_str
    if degree_map.get(name, 0) > SUPER_HUB_THRESHOLD:
        return False
    if type_str in SPECIFIC_TYPES:
        return True
    return False


def cluster_triples_v2(df, min_new_docs: int = 2, top_n: int = 100) -> list[list[dict]]:
    """
    Improved clustering:
    - Drops triples where BOTH ends are super-hubs
    - Prefers triples with at least one specific entity
    - Builds topic clusters (max 6 triples) avoiding domination by a single hub
    """
    filtered = df[df["new_docs_count"] >= min_new_docs].head(top_n).copy()

    # Parse the type from "Type: Name"
    def parse_type(s):
        return s.split(": ", 1)[0] if ": " in s else "?"

    filtered["head_type"] = filtered["head"].apply(parse_type)
    filtered["tail_type"] = filtered["tail"].apply(parse_type)

    # Score the specificity of each triple
    filtered["head_specific"] = filtered.apply(
        lambda r: is_specific(r["head"], r["head_type"]), axis=1
    )
    filtered["tail_specific"] = filtered.apply(
        lambda r: is_specific(r["tail"], r["tail_type"]), axis=1
    )
    filtered["specificity"] = filtered["head_specific"].astype(int) + filtered["tail_specific"].astype(int)

    # Drop triples where neither end is specific
    filtered = filtered[filtered["specificity"] >= 1]

    # Sort: specific first, then by new_docs_count
    filtered = filtered.sort_values(
        ["specificity", "new_docs_count"], ascending=False
    ).reset_index(drop=True)

    print(f"After filtering: {len(filtered)} triples (dropped super-hub-only ones)")

    # Cluster — max 1 super-hub per cluster, max 6 triples
    entity_to_triples = defaultdict(list)
    for _, row in filtered.iterrows():
        t = row.to_dict()
        entity_to_triples[row["head"]].append(t)
        entity_to_triples[row["tail"]].append(t)

    clusters = []
    used = set()

    for _, row in filtered.iterrows():
        key = (row["head"], row["relation"], row["tail"])
        if key in used:
            continue

        cluster = [row.to_dict()]
        used.add(key)

        # Count how many super-hubs are already in the cluster
        def hub_count(cl):
            hubs = set()
            for t in cl:
                for f in ("head", "tail"):
                    nm = t[f].split(": ", 1)[-1] if ": " in t[f] else t[f]
                    if degree_map.get(nm, 0) > SUPER_HUB_THRESHOLD:
                        hubs.add(nm)
            return len(hubs)

        neighbors = (
                entity_to_triples[row["head"]] +
                entity_to_triples[row["tail"]]
        )
        # Sort neighbors by specificity (specific first)
        neighbors_sorted = sorted(
            neighbors,
            key=lambda t: t.get("specificity", 0) + t.get("new_docs_count", 0),
            reverse=True
        )

        for neighbor in neighbors_sorted:
            nkey = (neighbor["head"], neighbor["relation"], neighbor["tail"])
            if nkey in used:
                continue
            if len(cluster) >= 6:
                break
            if hub_count(cluster + [neighbor]) > 2:
                continue
            cluster.append(neighbor)
            used.add(nkey)

        if len(cluster) >= 2:
            clusters.append(cluster)

    clusters.sort(
        key=lambda cl: sum(t.get("specificity", 0) for t in cl) / len(cl),
        reverse=True
    )

    return clusters


clusters = cluster_triples_v2(df, min_new_docs=2, top_n=100)
print(f"Generated {len(clusters)} clusters\n")

for i, cluster in enumerate(clusters[:5]):
    print(f"Cluster [{i}]  (specificity avg={sum(t.get('specificity',0) for t in cluster)/len(cluster):.1f})")
    for t in cluster:
        spec = "⭐" if t.get("specificity", 0) >= 1 else "  "
        print(f"  {spec} {t['head']} --[{t['relation']}]--> {t['tail']}  (new_docs={t['new_docs_count']})")
    print()

In [ ]:
import time, json

qa_pairs_v2 = []
errors_v2   = []

for i, cluster in enumerate(clusters):
    user_prompt = build_user_prompt(cluster)

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.9,       # higher temperature = more creative questions
            response_format={"type": "json_object"},
        )

        raw  = response.choices[0].message.content
        data = json.loads(raw)

        qa_pairs_v2.append({
            "id":               i,
            "question":         data["question"],
            "answer":           data["answer"],
            "documents_needed": data.get("documents_needed", []),
            "reasoning":        data.get("reasoning", ""),
            "triples":          cluster,
            "n_triples":        len(cluster),
            "n_new_docs":       sum(t["new_docs_count"] for t in cluster),
        })

        print(f"[{i+1:>2}/{len(clusters)}]  {data['question'][:90]}...")

    except Exception as e:
        errors_v2.append({"cluster_id": i, "error": str(e)})
        print(f"[{i+1:>2}/{len(clusters)}] ❌  {e}")

    time.sleep(0.3)

print(f"Generated {len(qa_pairs_v2)} pairs  |  {len(errors_v2)} errors")

In [ ]:
JUDGE_PROMPT_V2 = """You are evaluating whether a question truly requires graph-based retrieval (GraphRAG) vs simple keyword search (VanillaRAG).

Score 1-5:
1 = Vanilla RAG wins: single document or obvious keywords answer this
2 = Vanilla RAG likely answers: 1-2 documents, keywords overlap
3 = Toss-up: might need 2 docs but keywords partially help
4 = GraphRAG wins: requires 2-3 docs connected only via entity relationships
5 = GraphRAG clearly wins: 3+ docs, no keyword overlap, requires entity traversal

Key test: Can you answer this by searching for the main subject literally? If yes → score ≤ 2.

Return JSON: {"score": <int>, "reason": "<1 sentence explaining decision>", "keyword_that_would_cheat": "<word that naive search would use>"}"""

scored_v2 = []
for qa in qa_pairs_v2:
    try:
        r = client.chat.completions.create(
            model="gpt-5.1",
            messages=[
                {"role": "system", "content": JUDGE_PROMPT_V2},
                {"role": "user",   "content": f"Question: {qa['question']}\nAnswer: {qa['answer']}\nReasoning: {qa['reasoning']}"},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )
        d = json.loads(r.choices[0].message.content)
        scored_v2.append({
            **qa,
            "vanilla_difficulty":  d["score"],
            "judge_reason":        d["reason"],
            "cheating_keyword":    d.get("keyword_that_would_cheat", ""),
        })
        print(f"[score={d['score']}] {qa['question'][:70]}...")
        print(f"        ⚠ cheat keyword: '{d.get('keyword_that_would_cheat','')}' | {d['reason']}")
        time.sleep(0.2)
    except Exception as e:
        scored_v2.append({**qa, "vanilla_difficulty": 0, "judge_reason": str(e), "cheating_keyword": ""})

hard_v2 = [q for q in scored_v2 if q["vanilla_difficulty"] >= 4]
print(f"Hard for vanilla (score ≥ 4): {len(hard_v2)} / {len(scored_v2)}")
def flatten_answer(qa: dict) -> str:
    ans = qa.get("answer", "")
    if isinstance(ans, dict):
        parts = []
        if "fact_1" in ans:
            parts.append(ans["fact_1"].get("text", ""))
        if "fact_2" in ans:
            parts.append(ans["fact_2"].get("text", ""))
        if "synthesis" in ans:
            parts.append(ans["synthesis"])
        return " ".join(filter(None, parts))
    return str(ans)

for q in sorted(hard_v2, key=lambda x: -x["vanilla_difficulty"]):
    print(f"[score={q['vanilla_difficulty']}] {q['question']}")
    print(f"{flatten_answer(q)[:150]}...")
    print(f"{q['judge_reason']}")

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchAny

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "vectorstore"

qdrant = QdrantClient(url=QDRANT_URL)
print("Connected to Qdrant")

In [ ]:
CYPHER_CHUNK_IDS = """
MATCH (n)
WHERE elementId(n) = $node_id
RETURN n.chunk_ids AS chunk_ids
"""

def get_chunk_ids_for_node(session, node_id: str) -> list[str]:
    result = session.run(CYPHER_CHUNK_IDS, node_id=node_id).single()
    if result and result["chunk_ids"]:
        return list(result["chunk_ids"])
    return []


def get_node_id_by_name_type(session, name: str, type_: str) -> str | None:
    result = session.run(
        "MATCH (n {name: $name, type: $type}) RETURN elementId(n) AS eid LIMIT 1",
        name=name, type=type_
    ).single()
    return result["eid"] if result else None


all_nodes = {}  # {(name, type): node_id}

with driver.session() as session:
    for cluster in clusters:
        for triple in cluster:
            for field in ("head", "tail"):
                raw = triple[field]
                if ": " in raw:
                    tp, nm = raw.split(": ", 1)
                else:
                    tp, nm = "?", raw

                key = (nm, tp)
                if key not in all_nodes:
                    nid = get_node_id_by_name_type(session, nm, tp)
                    all_nodes[key] = nid

print(f"Unique nodes to check: {len(all_nodes)}")
print(f"Found in the graph: {sum(1 for v in all_nodes.values() if v)}")

node_chunk_ids = {}  # {(name, type): [chunk_ids]}

with driver.session() as session:
    for (nm, tp), nid in all_nodes.items():
        if nid:
            cids = get_chunk_ids_for_node(session, nid)
            node_chunk_ids[(nm, tp)] = cids
        else:
            node_chunk_ids[(nm, tp)] = []

print(f"Fetched chunk_ids for all nodes")

In [ ]:
def fetch_chunks_from_qdrant(chunk_ids: list[str]) -> list[dict]:
    """Fetches chunks from Qdrant by chunk_id. Returns a list of dicts with text and metadata."""
    if not chunk_ids:
        return []

    points, _ = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=Filter(
            must=[FieldCondition(
                key="metadata.chunk_id",
                match=MatchAny(any=chunk_ids)
            )]
        ),
        with_payload=True,
        with_vectors=False,
        limit=len(chunk_ids) + 10,
    )

    results = []
    for p in points:
        payload  = p.payload or {}
        metadata = payload.get("metadata", {})
        content  = payload.get("page_content", "")
        section  = metadata.get("section", "")
        content  = content.replace(f"{section}\n", "").strip()

        results.append({
            "chunk_id":      metadata.get("chunk_id", ""),
            "document_name": metadata.get("document_name", ""),
            "section":       section,
            "created_at":    metadata.get("created_at", ""),
            "text":          content,
        })

    return results


test_key = list(node_chunk_ids.keys())[0]
test_ids = node_chunk_ids[test_key][:3]
test_chunks = fetch_chunks_from_qdrant(test_ids)
print(f"Test for node '{test_key[0]}': {len(test_chunks)} chunks fetched from Qdrant")
if test_chunks:
    print(f"{test_chunks[0]['document_name']} / {test_chunks[0]['section']}")
    print(f"{test_chunks[0]['text'][:100]}...")

In [ ]:
def enrich_qa_with_chunks(qa_pairs: list[dict], node_chunk_ids: dict) -> list[dict]:
    """
    For each QA pair collects source chunks from all nodes in the triples.
    Deduplicates by chunk_id and groups per document.
    """
    enriched = []

    for qa in qa_pairs:
        all_chunk_ids = set()
        node_sources  = []  # [{node, chunk_ids}]

        for triple in qa["triples"]:
            for field in ("head", "tail"):
                raw = triple[field]
                tp, nm = raw.split(": ", 1) if ": " in raw else ("?", raw)
                key    = (nm, tp)
                cids   = node_chunk_ids.get(key, [])

                if cids:
                    node_sources.append({
                        "node":      nm,
                        "type":      tp,
                        "chunk_ids": cids,
                    })
                    all_chunk_ids.update(cids)

        chunks = fetch_chunks_from_qdrant(list(all_chunk_ids))
        docs_map = defaultdict(list)
        for chunk in chunks:
            docs_map[chunk["document_name"]].append(chunk)

        enriched.append({
            **qa,
            "source_chunks": chunks,
            "source_documents": [
                {
                    "document_name": doc_name,
                    "chunks": [
                        {
                            "chunk_id": c["chunk_id"],
                            "section":  c["section"],
                            "text":     c["text"],
                        }
                        for c in doc_chunks
                    ]
                }
                for doc_name, doc_chunks in docs_map.items()
            ],
            "n_source_chunks":    len(chunks),
            "n_source_documents": len(docs_map),
        })

        print(
            f"  QA [{qa['id']}]: {len(chunks)} chunks from {len(docs_map)} documents"
            f"  | '{qa['question'][:60]}...'"
        )

    return enriched


print("Enriching QA with source context...\n")
qa_enriched_v2 = enrich_qa_with_chunks(scored_v2, node_chunk_ids)
print(f"\nDone. {len(qa_enriched_v2)} QA pairs with context.")

In [ ]:
# Full JSON with everything
with open("graph_qa_with_context_2.json", "w", encoding="utf-8") as f:
    json.dump(qa_enriched_v2, f, ensure_ascii=False, indent=2)

# Light version (without full chunk texts — chunk_ids only)
qa_light = []
for qa in qa_enriched_v2:
    qa_light.append({
        "id":               qa["id"],
        "question":         qa["question"],
        "answer":           qa["answer"],
        "reasoning":        qa["reasoning"],
        "judge_reason":     qa["judge_reason"],
        "n_triples":        qa["n_triples"],
        "n_new_docs":       qa["n_new_docs"],
        "n_source_chunks":  qa["n_source_chunks"],
        "n_source_documents": qa["n_source_documents"],
        "source_documents": [
            {
                "document_name": d["document_name"],
                "chunks": [
                    {
                        "chunk_id": c["chunk_id"],
                        "section":  c["section"],
                        "text":     c["text"]
                    }
                    for c in d["chunks"]
                ]
            }
            for d in qa["source_documents"]
        ],
        "triples": [
            f"{t['head']} --[{t['relation']}]--> {t['tail']}"
            for t in qa["triples"]
        ],
    })

with open("graph_qa_light_2.json", "w", encoding="utf-8") as f:
    json.dump(qa_light, f, ensure_ascii=False, indent=2)

print("   Saved:")
print("   graph_qa_with_context_2.json  — full (with chunk texts)")
print("   graph_qa_light_2.json         — light (without texts, chunk_ids only)")


total_chunks = sum(q["n_source_chunks"] for q in qa_enriched_v2)
total_docs   = sum(q["n_source_documents"] for q in qa_enriched_v2)
print(f"\n Statistics:")
print(f"   QA pairs:            {len(qa_enriched_v2)}")
print(f"   Chunks total:        {total_chunks}  (avg {total_chunks/len(qa_enriched_v2):.1f} / question)")
print(f"   Documents total:     {total_docs}    (avg {total_docs/len(qa_enriched_v2):.1f} / question)")

In [ ]:
import pandas as pd
import json

with open("graph_qa_with_context_2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for entry in data:
    contexts = []
    for doc in entry.get("source_documents", []):
        for chunk in doc.get("chunks", []):
            text = chunk.get("text", "").strip()
            if text:
                contexts.append(f"[{doc['document_name']}] {text}")

    rows.append({
        "question": entry.get("question", ""),
        "reference_contexts": "\n\n".join(contexts),
        "answer": entry.get("answer", "")
    })

df = pd.DataFrame(rows, columns=["question", "reference_contexts", "answer"])

output_path = "graph_qa_eval_3.ods"
df.to_excel(output_path, index=False, engine="odf")

print(f"Saved {len(df)} rows -> {output_path}")
df[["question", "answer"]].head(3)